<a href="https://www.kaggle.com/code/aryanbhattarai/unet-seg-layer?scriptVersionId=333022614" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torchvision.models as models
import os

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from PIL import Image
from tqdm import tqdm
import albumentations as augmentation
from albumentations.pytorch import ToTensorV2
from sklearn.model_selection import train_test_split
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms as transforms
import torchvision.models as models

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
root_dir = "/kaggle/input/multichannel-glaucoma-benchmark-dataset"
metadata = pd.read_csv(os.path.join(root_dir, "metadata - standardized.csv"))

In [ ]:
metadata = metadata[
    ~metadata['fundus_oc_seg'].isnull() &
    (metadata['fundus_oc_seg'] != 'Not Visible') &
    ~metadata['fundus_od_seg'].isnull()
].reset_index(drop=True)

# Preprocessing copied from "preprocessing" notebook

In [ ]:
# === 1. Imports ===

import cv2
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import argrelextrema
from scipy.stats import mode
from skimage import measure

In [ ]:
# === 2. Utility Functions ===


def show(img, title=""):
    plt.figure(figsize=(4,4))
    if len(img.shape) == 2:
        plt.imshow(img, cmap='gray')
    else:
        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        plt.title(title)
        plt.axis('off')

In [ ]:
# === 3. Clahe Preprocessing (Paper Section III) ===

# why the cliplimit of 2??

def apply_clahe(img):
    """
    Apply CLAHE on the green channel (standard practice for fundus images)
    """
    green = img[:, :, 1]
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    enhanced = clahe.apply(green)
    return enhanced

In [ ]:
# === 4. Dynamic Histogram-Based Threshold (Paer Section IV) ===

def compute_dynamic_threshold(gray):
    """
    Approximation of paper's histogram-extrema-based thresholding
    """
    hist = cv2.calcHist([gray],[0],None,[256],[0,256]).flatten()
    hist = np.log1p(hist) # log normalization
    
    
    # smooth histogram
    hist_smooth = cv2.GaussianBlur(hist, (9,1), 0).flatten()
    
    
    # find local minima and maxima
    local_max = argrelextrema(hist_smooth, np.greater)[0]
    local_min = argrelextrema(hist_smooth, np.less)[0]
    
    
    # restrict search space (paper uses t1=192, t2=128)
    local_max = [x for x in local_max if x <= 192]
    local_min = [x for x in local_min if x <= 128]
    
    
    best_thresh = 0
    best_score = 0
    
    
    for m1 in local_max:
        for mn in local_min:
            for m2 in local_max:
                if m1 < mn < m2:
                    score = abs(hist_smooth[m1] - hist_smooth[mn]) \
                    + abs(hist_smooth[mn] - hist_smooth[m2])
                    if score > best_score:
                        best_score = score
                        best_thresh = mn
    
    
    return best_thresh

In [ ]:
# === 5. Foreground Segmentation ===

def segment_foreground(gray):
    thresh = compute_dynamic_threshold(gray)
    _, mask = cv2.threshold(gray, thresh, 255, cv2.THRESH_BINARY)
    
    
    # keep largest connected component
    labels = measure.label(mask)
    props = measure.regionprops(labels)
    if len(props) == 0:
        return mask
    
    
    largest = max(props, key=lambda x: x.area)
    clean_mask = np.zeros_like(mask)
    clean_mask[labels == largest.label] = 255
    
    
    return clean_mask

In [ ]:
# === 6. Fundus Center Estimation (Paper Algorithm 2) ===

def estimate_center(mask):
    h, w = mask.shape
    row_medians = []
    col_medians = []
    
    
    for i in range(h):
        cols = np.where(mask[i] > 0)[0]
        if len(cols) > 0:
            row_medians.append(np.median(cols))
    
    
    for j in range(w):
        rows = np.where(mask[:, j] > 0)[0]
        if len(rows) > 0:
            col_medians.append(np.median(rows))
    
    
    cx = int(mode(row_medians, keepdims=True).mode[0])
    cy = int(mode(col_medians, keepdims=True).mode[0])
    
    
    return cx, cy

In [ ]:
# === 7. Fundus Radius Estimation (paper Algorithm 3) ===

def estimate_radius(mask, cx, cy):
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    if len(contours) == 0:
        return min(mask.shape) // 2
    

    largest = max(contours, key=cv2.contourArea)
    dists = []
    for p in largest:
        x, y = p[0]
        dists.append(np.sqrt((x-cx)**2 + (y-cy)**2))
    
    
    return int(mode(np.round(dists), keepdims=True).mode[0])

In [ ]:
# === 8. Crop + Pad + Resize Standardization ===

def standardize_fundus(img, size=512):
    gray = apply_clahe(img)
    mask = segment_foreground(gray)
    
    
    cx, cy = estimate_center(mask)
    r = estimate_radius(mask, cx, cy)
    
    
    h, w = img.shape[:2]
    
    
    x1 = max(cx - r, 0)
    x2 = min(cx + r, w)
    y1 = max(cy - r, 0)
    y2 = min(cy + r, h)
    
    
    cropped = img[y1:y2, x1:x2]
    
    
    # pad to square
    hh, ww = cropped.shape[:2]
    pad_top = pad_bottom = pad_left = pad_right = 0
    
    
    if hh > ww:
        diff = hh - ww
        pad_left = diff // 2
        pad_right = diff - pad_left
    else:
        diff = ww - hh
        pad_top = diff // 2
        pad_bottom = diff - pad_top
    
    
    padded = cv2.copyMakeBorder(
    cropped, pad_top, pad_bottom, pad_left, pad_right,
    cv2.BORDER_CONSTANT, value=0
    )
    
    
    standardized = cv2.resize(padded, (size, size), interpolation=cv2.INTER_AREA)
    return standardized

In [ ]:
# === 9. Full Pipeline Wrapper ===

def preprocess_fundus_image(path, size=512, show_steps=False):
    img = cv2.imread(path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    
    if show_steps:
        show(img, "Original")
    
    
    out = standardize_fundus(img, size)
    
    
    if show_steps:
        show(out, "Standardized")
    
    
    return out

# Making Dataset


In [ ]:
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms as transforms

In [ ]:
class FundusSegmentationDataset(Dataset):
    def __init__(self, metadata, root_dir, image_size=128, augment=False):
        self.metadata = metadata
        self.root_dir = root_dir
        self.image_size = image_size
        self.augment = augment

        self.aug_transform = augmentation.Compose([
            augmentation.Resize(image_size, image_size),
            augmentation.HorizontalFlip(p=0.5),
            augmentation.RandomBrightnessContrast(p=0.2),
            augmentation.Affine(scale=(0.95, 1.05), translate_percent=(0.02, 0.02), rotate=(-15, 15), p=0.5),
            augmentation.GaussianBlur(p=0.1),
            augmentation.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),  # normalize to mean=0, std=1
            ToTensorV2(),
        ])

        self.simple_transform = augmentation.Compose([
            augmentation.Resize(image_size, image_size),
            augmentation.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),  # normalize to mean=0, std=1
            ToTensorV2(),
        ])

    def __len__(self):
        return len(self.metadata)

    def __getitem__(self, idx):
        row = self.metadata.iloc[idx]
        fundus_path = os.path.join(self.root_dir, "full-fundus" + row['fundus'])
        od_path = os.path.join(self.root_dir, "optic-disc" + row['fundus_od_seg'])
        oc_path = os.path.join(self.root_dir, "optic-cup" + row['fundus_oc_seg'])
    
        # === Applying Preprocessing ===
        raw_image = np.array(Image.open(fundus_path).convert("RGB"))
        # fundus = preprocess_fundus_image(raw_image)
        fundus = preprocess_fundus_image(fundus_path)
        
        od = np.array(Image.open(od_path))
        oc = np.array(Image.open(oc_path))
    
        mask = np.zeros_like(od)
        mask[od != 0] = 2  # OD = 2
        mask[oc != 0] = 1  # OC = 1
    
        if self.augment:
            transformed = self.aug_transform(image=fundus, mask=mask)
        else:
            transformed = self.simple_transform(image=fundus, mask=mask)

        # albumentations returns a dictionary. 
        # ex: transformed{
        #     "image": transformed_image,
        #     "mask": transformed_mask
        # }
        fundus = transformed['image']
        mask = transformed['mask'].long()
    
        return fundus, mask

# Convolution Block
this is the same as upconv but this doesnt change the spacial resolution. 

In [ ]:
# class ConvBlock(nn.Module):
#     """Double convolution block"""
#     def __init__(self, in_ch, out_ch, kernel=3):
#         super().__init__()
#         self.conv = nn.Sequential(
#             nn.Conv2d(in_ch, out_ch, kernel, padding=1),
#             nn.BatchNorm2d(out_ch),
#             nn.ReLU(inplace=True),
#         )
    
#     def forward(self, x):
#         return self.conv(x)

In [ ]:
class ConvBlock(nn.Module):
    """Double convolution block"""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )
    
    def forward(self, x):
        return self.conv(x)

# Up convolution Block
this also changes the spacial resolution. 

In [ ]:
class UpConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.up = nn.Sequential(
            nn.Upsample(scale_factor=2),
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.up(x)

# Attention Block

In [ ]:
class AttentionBlock(nn.Module):
    def __init__(self, F_g, F_l, F_int):
        super().__init__()
        self.W_g = nn.Sequential(nn.Conv2d(F_g, F_int, 1), nn.BatchNorm2d(F_int))
        self.W_x = nn.Sequential(nn.Conv2d(F_l, F_int, 1), nn.BatchNorm2d(F_int))
        self.psi = nn.Sequential(nn.Conv2d(F_int, 1, 1), nn.BatchNorm2d(1), nn.Sigmoid())

        
    def forward(self, g, x):
        g1 = self.W_g(g)
        x1 = self.W_x(x)

        combined = F.relu(g1 + x1)
        attention_mask = self.psi(combined)

        return x * attention_mask

# Standard Unet


In [ ]:
class StandardUNet(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()

        # === using the resnet50 model ===
        resnet = models.resnet50(weights='DEFAULT')

        # Encoder
        self.enc0 = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu)  # 64 channels
        self.mp = resnet.maxpool 
        self.enc1 = resnet.layer1  # 256 channels
        self.enc2 = resnet.layer2  # 512 channels
        self.enc3 = resnet.layer3  # 1024 channels
        self.center = resnet.layer4  # 2048 channels

        # Decoder with Concatenation
        # 1. Upsample the lower layer
        # 2. Concatenate with the skip connection
        # 3. Apply a convolution to merge them

        # Decoder 
        self.up4 = UpConv(2048, 1024) 
        self.conv4 = ConvBlock(1024 + 1024, 1024)
        
        self.up3 = UpConv(1024, 512)
        self.conv3 = ConvBlock(512 + 512, 512)
        
        self.up2 = UpConv(512, 256)
        self.conv2 = ConvBlock(256 + 256, 256)
        
        self.up1 = UpConv(256, 64)
        self.conv1 = ConvBlock(64 + 64, 64)

        self.final = ConvBlock(64, num_classes) #upsampling also needed in here

    def forward(self, x):
        # Encoder
        e0 = self.enc0(x); # 64
        mxp = self.mp(e0)
        e1 = self.enc1(mxp);  # 256
        e2 = self.enc2(e1);  # 512
        e3 = self.enc3(e2)  # 1024
        c = self.center(e3)  # 2048
        
        # Decoder
        d4 = self.up4(c)
        d4 = torch.cat([d4, e3], dim=1) # Concatenate along channels
        d4 = self.conv4(d4)
        
        d3 = self.up3(d4)
        d3 = torch.cat([d3, e2], dim=1)
        d3 = self.conv3(d3)
        
        d2 = self.up2(d3)
        d2 = torch.cat([d2, e1], dim=1)
        d2 = self.conv2(d2)
        
        d1 = self.up1(d2)
        d1 = torch.cat([d1, e0], dim=1)
        d1 = self.conv1(d1)
        
        return self.final(d1)

# Attention Unet

In [ ]:
class AttentionUNet(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()

        # === using the resnet50 model ===
        resnet = models.resnet50(weights='DEFAULT')

        # Encoder
        self.enc0 = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu)  # 64 channels
        self.mp = resnet.maxpool
        self.enc1 = resnet.layer1  # 256 channels
        self.enc2 = resnet.layer2  # 512 channels
        self.enc3 = resnet.layer3  # 1024 channels
        self.center = resnet.layer4  # 2048 channels

        # Decoder with Concatenation
        # 1. Upsample the lower layer
        # 2. Concatenate with the skip connection
        # 3. Apply a convolution to merge them

        # Decoder 
        self.up4 = UpConv(2048, 1024) 
        self.att4 = AttentionBlock(F_g=1024, F_l=1024, F_int=512)
        self.conv4 = ConvBlock(1024 + 1024, 1024)
        
        self.up3 = UpConv(1024, 512)
        self.att3 = AttentionBlock(F_g=512, F_l=512, F_int=256)
        self.conv3 = ConvBlock(512 + 512, 512)
        
        self.up2 = UpConv(512, 256)
        self.att2 = AttentionBlock(F_g=256, F_l=256, F_int=128)
        self.conv2 = ConvBlock(256 + 256, 256)
        
        self.up1 = UpConv(256, 64)
        self.att1 = AttentionBlock(F_g=64, F_l=64, F_int=32)
        self.conv1 = ConvBlock(64 + 64, 64)

        self.final = ConvBlock(64, num_classes) #upsampling needed here too

    def forward(self, x):
        # Encoder
        e0 = self.enc0(x); # 64
        mxp = self.mp(e0)
        e1 = self.enc1(mxp);  # 256
        e2 = self.enc2(e1);  # 512
        e3 = self.enc3(e2)  # 1024
        c = self.center(e3)  # 2048
        
        # Decoder
        d4 = self.up4(c)
        e3_att = self.att4(g=d4, x=e3)            # 1024
        d4 = torch.cat([d4, e3_att], dim=1)    # Concatenate along channels
        d4 = self.conv4(d4)
        
        d3 = self.up3(d4)
        e2_att = self.att3(g=d3, x=e2)            # 512
        d3 = torch.cat([d3, e2_att], dim=1)
        d3 = self.conv3(d3)
        
        d2 = self.up2(d3)
        e1_att = self.att2(g=d2, x=e1)            # 256
        d2 = torch.cat([d2, e1_att], dim=1)
        d2 = self.conv2(d2)
        
        d1 = self.up1(d2)
        e0_att = self.att1(g=d1, x=e0)            # 64
        d1 = torch.cat([d1, e0_att], dim=1)
        d1 = self.conv1(d1)
        
        return self.final(d1)

# Dimension Visualization

In [ ]:
def shape_hook(name):
    def hook(module, input, output):
        if isinstance(output, (list, tuple)):
            out = output[0]
        else:
            out = output
        print(f"{name:<15} -> shape: {tuple(out.shape)}")
    return hook

def register_hooks(model):
    model.enc0.register_forward_hook(shape_hook("enc0"))
    model.mp.register_forward_hook(shape_hook("mp"))
    model.enc1.register_forward_hook(shape_hook("enc1"))
    model.enc2.register_forward_hook(shape_hook("enc2"))
    model.enc3.register_forward_hook(shape_hook("enc3"))
    model.center.register_forward_hook(shape_hook("center"))

    model.up4.register_forward_hook(shape_hook("up4"))
    model.conv4.register_forward_hook(shape_hook("conv4"))

    model.up3.register_forward_hook(shape_hook("up3"))
    model.conv3.register_forward_hook(shape_hook("conv3"))

    model.up2.register_forward_hook(shape_hook("up2"))
    model.conv2.register_forward_hook(shape_hook("conv2"))

    model.up1.register_forward_hook(shape_hook("up1"))
    model.conv1.register_forward_hook(shape_hook("conv1"))

    model.final.register_forward_hook(shape_hook("final"))


def register_attention_hooks(model):
    # Encoder
    model.enc0.register_forward_hook(shape_hook("enc0"))
    model.mp.register_forward_hook(shape_hook("mp"))
    model.enc1.register_forward_hook(shape_hook("enc1"))
    model.enc2.register_forward_hook(shape_hook("enc2"))
    model.enc3.register_forward_hook(shape_hook("enc3"))
    model.center.register_forward_hook(shape_hook("center"))

    # Decoder upsampling
    model.up4.register_forward_hook(shape_hook("up4"))
    model.up3.register_forward_hook(shape_hook("up3"))
    model.up2.register_forward_hook(shape_hook("up2"))
    model.up1.register_forward_hook(shape_hook("up1"))

    # Attention blocks
    model.att4.register_forward_hook(shape_hook("att4_out"))
    model.att3.register_forward_hook(shape_hook("att3_out"))
    model.att2.register_forward_hook(shape_hook("att2_out"))
    model.att1.register_forward_hook(shape_hook("att1_out"))

    # Decoder convs
    model.conv4.register_forward_hook(shape_hook("conv4"))
    model.conv3.register_forward_hook(shape_hook("conv3"))
    model.conv2.register_forward_hook(shape_hook("conv2"))
    model.conv1.register_forward_hook(shape_hook("conv1"))

    # Final
    model.final.register_forward_hook(shape_hook("final"))


In [ ]:
import torch

# Create model
model = StandardUNet(num_classes=3)
model.eval()

# Register hooks
register_hooks(model)

# Dummy input image (change size if you want)
x = torch.randn(1, 3, 512, 512)

print(f"{'input':<15} -> shape: {tuple(x.shape)}")
print("-" * 50)

# Forward pass
with torch.no_grad():
    _ = model(x)


In [ ]:
import torch

model = AttentionUNet(num_classes=3)
model.eval()

register_attention_hooks(model)

x = torch.randn(1, 3, 512, 512)

print(f"{'input':<15} -> shape: {tuple(x.shape)}")
print("-" * 60)

with torch.no_grad():
    _ = model(x)


# Dice function

In [ ]:
# === DICE FUNCTION ===
def dice_score(pred, target, class_idx):
    pred = (pred == class_idx).float()
    target = (target == class_idx).float()
    
    numerator = 2 * (pred * target).sum()
    denominator = pred.sum() + target.sum() + 1e-6 # <=== 1e-6 to prevent divide by zero error
    return numerator / denominator

# Splits

In [ ]:
# Manual split index to control augmentation only in training
indices = np.arange(len(metadata))
np.random.seed(42)
np.random.shuffle(indices)
train_idx = indices[:-200]
val_idx = indices[-200:-100]
test_idx = indices[-100:]

# Initiliazing datasets

In [ ]:
# Dataset with augmentation is for training only
train_set = FundusSegmentationDataset(metadata.iloc[train_idx], root_dir, augment=True)
val_set   = FundusSegmentationDataset(metadata.iloc[val_idx], root_dir, augment=False)
test_set  = FundusSegmentationDataset(metadata.iloc[test_idx], root_dir, augment=False)

# Initiliazing dataloaders

In [ ]:
# DataLoaders
train_loader = DataLoader(train_set, batch_size=8, shuffle=True)
val_loader   = DataLoader(val_set, batch_size=8)
test_loader  = DataLoader(test_set, batch_size=8)

# Training Loop

In [ ]:
# === Training Loop ===
def train_with_earlystop(model, train_loader, val_loader, epochs=500, patience=50):
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10)
    criterion = nn.CrossEntropyLoss()

    history = {
        'train_loss': [],
        'val_loss': [],
        'dice_oc': [],
        'dice_od': []
    }
    best_val_loss = float('inf')
    patience_counter = 0

    model.to(device)

    for epoch in range(epochs):
        model.train()
        train_loss = 0
        
        for img, mask in train_loader:
            img, mask = img.to(device), mask.to(device)
            pred = model(img)
            pred = F.interpolate(pred,
                                 size=mask.shape[-2:],
                                 mode='bilinear',
                                 align_corners=False) #this resizes the pred to match dimension of mask. can be removed later???
            loss = criterion(pred, mask)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            train_loss += loss.item()


        model.eval()
        val_loss = 0
        dices = {
            'oc': [],
            'od': []
        }
        with torch.no_grad():
            for img, mask in val_loader:
                img, mask = img.to(device), mask.to(device)
                pred = model(img)
                pred = F.interpolate(pred,
                                     size=mask.shape[-2:],
                                     mode='bilinear',
                                     align_corners=False)
                loss = criterion(pred, mask)
                val_loss += loss.item()
                pred_label = torch.argmax(pred, dim=1)
                dices['oc'].append(dice_score(pred_label, mask, 1).item())
                dices['od'].append(dice_score(pred_label, mask, 2).item())

        avg_train_loss = train_loss / len(train_loader)
        avg_val_loss = val_loss / len(val_loader)
        mean_oc = np.mean(dices['oc'])
        mean_od = np.mean(dices['od'])

        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(avg_val_loss)
        history['dice_oc'].append(mean_oc)
        history['dice_od'].append(mean_od)

        print(f"Epoch {epoch+1:02d} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Dice OC: {mean_oc:.4f} | Dice OD: {mean_od:.4f}")

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), "attention_unet_best_v2.pt")
            patience_counter = 0
        else:
            patience_counter += 1
            print(f"⏳ Early stopping patience: {patience_counter}/{patience}")
            if patience_counter >= patience:
                print("🛑 Early stopping triggered.")
                break

    return history



# Model Initialization

In [ ]:
# Model Initialization
model = AttentionUNet(num_classes=3)
history = train_with_earlystop(model, train_loader, val_loader, epochs=75)

In [ ]:
# Plot Metric
plt.figure(figsize=(10,4))
plt.subplot(1,2,1); plt.plot(history['train_loss'], label='Train')
plt.plot(history['val_loss'], label='Val')
plt.title("Loss")
plt.legend()
plt.subplot(1,2,2); plt.plot(history['dice_oc'], label='Dice OC')
plt.plot(history['dice_od'], label='Dice OD')
plt.title("Dice Score")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
def evaluate_and_visualize(model, test_loader, num_samples=4):
    """
    Mengevaluasi model pada data test dan menampilkan hasil Dice Score serta visualisasi prediksi vs ground truth.
    """
    model.eval()
    model.to(device)

    dice_oc_scores = []
    dice_od_scores = []
    samples = []

    with torch.no_grad():
        for i, (img, mask) in enumerate(test_loader):
            img, mask = img.to(device), mask.to(device)
            pred = model(img)
            pred = F.interpolate(pred, size=mask.shape[-2:], mode='bilinear', align_corners=False)
            pred_label = torch.argmax(pred, dim=1)

            dice_oc_scores.append(dice_score(pred_label, mask, 1).item())
            dice_od_scores.append(dice_score(pred_label, mask, 2).item())

            if len(samples) < num_samples:
                for j in range(min(len(img), num_samples - len(samples))):
                    samples.append((img[j].cpu(), mask[j].cpu(), pred_label[j].cpu()))

    print(f"🧪 Test Dice Score - Optic Cup (OC): {np.mean(dice_oc_scores):.4f}")
    print(f"🧪 Test Dice Score - Optic Disc (OD): {np.mean(dice_od_scores):.4f}")


    def denormalize(tensor):
        """Denormalize image tensor from ImageNet normalization"""
        mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
        img = tensor * std + mean
        img = torch.clamp(img, 0, 1)  # Ensure values are in [0, 1]
        return img.permute(1, 2, 0).numpy()
    
    def decode_mask(mask_tensor):
        mask = mask_tensor.numpy()
        vis = np.zeros((mask.shape[0], mask.shape[1], 3), dtype=np.uint8)
        vis[mask == 1] = [255, 0, 0]   # OC = merah
        vis[mask == 2] = [0, 255, 0]   # OD = hijau
        return vis

    fig, axs = plt.subplots(num_samples, 3, figsize=(12, 4 * num_samples))
    for i, (img, true_mask, pred_mask) in enumerate(samples):
        # axs[i, 0].imshow(img.permute(1, 2, 0))
        axs[i, 0].imshow(denormalize(img))
        axs[i, 0].set_title("Fundus Image")
        axs[i, 1].imshow(decode_mask(true_mask))
        axs[i, 1].set_title("Ground Truth Mask")
        axs[i, 2].imshow(decode_mask(pred_mask))
        axs[i, 2].set_title("Predicted Mask")
        for ax in axs[i]: ax.axis('off')

    plt.tight_layout()
    plt.show()


In [ ]:
# === Evaluation ===
model = AttentionUNet(num_classes=3)
model.load_state_dict(torch.load("attention_unet_best_v2.pt"))
evaluate_and_visualize(model, test_loader, num_samples=4)